## 1. The Single-Sample Loss Function

Since we are only evaluating one row at a time, we do not average the errors across the entire dataset. For a randomly selected sample $i$, the loss function becomes the squared error of that single point:

$$L_i(W, b) = (y_i - \hat{y}_i)^2$$

Where the prediction for this single patient is:

$$\hat{y}_i = b + w_1x_{i1} + w_2x_{i2} + \dots + w_m x_{im} = X_i W + b$$

## 2. Calculating the Gradients (Partial Derivatives)

We apply the Chain Rule to our single-sample loss function. Because there is no summation ($\sum$) over $N$ rows, the math simplifies significantly.

### A. Derivative with respect to the Intercept ($b$):

$$\frac{\partial L_i}{\partial b} = 2(y_i - \hat{y}_i) \cdot (-1)$$
$$\frac{\partial L_i}{\partial b} = -2(y_i - \hat{y}_i)$$

### B. Derivative with respect to a single Weight ($w_j$):

For the weight corresponding to feature $j$ of this specific patient:

$$\frac{\partial L_i}{\partial w_j} = 2(y_i - \hat{y}_i) \cdot (-x_{ij})$$
$$\frac{\partial L_i}{\partial w_j} = -2x_{ij}(y_i - \hat{y}_i)$$

In fully vectorized form for all 10 features of patient $i$:

$$\nabla_W L_i = -2 X_i^T (y_i - \hat{y}_i)$$

## 3. The Update Rule

The weights are updated immediately after evaluating that single sample $i$:

$$b_{new} = b_{old} - \eta \cdot \left[ -2(y_i - \hat{y}_i) \right]$$
$$W_{new} = W_{old} - \eta \cdot \left[ -2 X_i^T (y_i - \hat{y}_i) \right]$$

## The Geometric and Practical Difference

| Feature               | Batch Gradient Descent                         | Stochastic Gradient Descent (SGD)                                                                                                |
| :-------------------- | :--------------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------- |
| Data per Step         | Entire dataset ($N$ rows)                      | Exactly 1 random row                                                                                                             |
| Path to Minimum       | Smooth, direct, mathematical trajectory straight down the center of the bowl. | Violent, zigzagging, noisy trajectory. Because individual patients have noise, some steps move away from the true minimum. |
| Speed per Iteration   | Very slow for large data.                      | Fast. Takes only one vector multiplication per step.                                                                             |
| Convergence           | Stops exactly at the global minimum.           | Never truly stops; it violently bounces around the minimum because of single-row variance.                                       |

# Custom stochastic gradient class

In [4]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# data preparation
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
class StochasticGradientDescent:
  def __init__(self, learning_rate=0.01, epochs=100):
    self.lr = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self, X_train, y_train):
    N, m = X_train.shape
    self.coef_ = np.zeros(m)
    self.intercept_ = 0
    for epoch in range(self.epochs):
      # i am shuffling this dataset at the start of every epoch for true randomness
      indices = np.arange(N)
      np.random.shuffle(indices)
      X_shuffled = X_train[indices]
      y_shuffled = y_train[indices]

      # looping through each individual patient record one by one
      for i in range(N):
        X_i = X_shuffled[i]
        y_i = y_shuffled[i]

        #1. single sample prediction : scalar dot product + b
        y_pred_i = np.dot(X_i, self.coef_) + self.intercept_

        # calculate single sample error
        error_i = y_pred_i - y_i

        dl_db = 2 * error_i
        # gradients for this single sample
        dl_dw = 2 * error_i * X_i

        # update rule
        self.intercept_ = self.intercept_ - (self.lr * dl_db)
        self.coef_ = self.coef_ - (self.lr * dl_dw)

  def predict(self, X_test):
    return np.dot(X_test, self.coef_) + self.intercept_ # Corrected: changed * to +

In [14]:
sgd = StochasticGradientDescent(learning_rate=0.005, epochs=150)
sgd.fit(X_train_scaled, y_train)

y_pred_sgd = sgd.predict(X_test_scaled)
r2_sgd = r2_score(y_test, y_pred_sgd)

print("--- CUSTOM STOCHASTIC GRADIENT DESCENT ---")
print(f"R2 Score:  {r2_sgd:.6f}")
print(f"Intercept: {sgd.intercept_:.6f}")
print(f"All Coefficients: {sgd.coef_}")

--- CUSTOM STOCHASTIC GRADIENT DESCENT ---
R2 Score:  0.437951
Intercept: 157.317224
All Coefficients: [  5.06253448 -11.81395836  22.91249367  16.94801179 -45.20656532
  24.82566356   6.64706685  13.70745135  33.67501056   2.09027866]


## Code & Performance Analysis
If you run this code, you will find that the $R^2$ score is incredibly close to our Batch Gradient Descent and Scikit-Learn targets.

However, notice the fundamental shift in training behavior:

- In Batch GD, if we ran epochs=500 on our 353 training rows, the weights were updated exactly 500 times.
- In Stochastic GD, running epochs=150 updates the weights $150 \times 353 =$ 52,950 times!

Because it updates so frequently, SGD converges to a good answer in far fewer epochs, but it shakes and jitters the entire way down the loss bowl.